In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn


class WideBasicBlock(nn.Module):
    out_channels: int
    stride: int = 1
    dropout_rate: float = 0.0
    use_projection: bool = False

    @nn.compact
    def __call__(self, x, train=False):
        # Preserve the shortcut path, projecting it when width or spatial size changes.
        shortcut = x
        if self.use_projection:
            shortcut = nn.Conv(
                self.out_channels,
                (1, 1),
                strides=(self.stride, self.stride),
                use_bias=False,
                name='shortcut',
            )(x)

        # Run the widened pre-activation residual branch.
        y = nn.BatchNorm(use_running_average=not train, name='bn1')(x)
        y = nn.relu(y)
        y = nn.Conv(
            self.out_channels,
            (3, 3),
            strides=(self.stride, self.stride),
            padding='SAME',
            use_bias=False,
            name='conv1',
        )(y)
        y = nn.BatchNorm(use_running_average=not train, name='bn2')(y)
        y = nn.relu(y)
        if self.dropout_rate > 0:
            y = nn.Dropout(
                rate=self.dropout_rate,
                name='dropout',
            )(y, deterministic=not train)
        y = nn.Conv(
            self.out_channels,
            (3, 3),
            padding='SAME',
            use_bias=False,
            name='conv2',
        )(y)

        # Merge shortcut and residual features.
        y = y + shortcut
        return y


class WideNet(nn.Module):
    depth: int = 28
    widen_factor: int = 10
    dropout_rate: float = 0.0
    num_classes: int = 10

    @nn.compact
    def __call__(self, x, train=False):
        # Configure the WRN-28-10 channel plan for CIFAR-size inputs.
        if (self.depth - 4) % 6 != 0:
            raise ValueError("WideNet depth must satisfy depth = 6n + 4.")
        block_count = (self.depth - 4) // 6
        widths = [
            16,
            16 * self.widen_factor,
            32 * self.widen_factor,
            64 * self.widen_factor,
        ]

        # Convert image input into low-level features: (batch, 32, 32, 3) -> (batch, 32, 32, 16).
        x = nn.Conv(
            widths[0],
            (3, 3),
            padding='SAME',
            use_bias=False,
            name='conv1',
        )(x)

        # Run widened residual stages: 160, 320, then 640 channels for WRN-28-10.
        x = self._stage(
            x,
            widths[1],
            block_count,
            stride=1,
            train=train,
            name='layer1',
        )
        x = self._stage(
            x,
            widths[2],
            block_count,
            stride=2,
            train=train,
            name='layer2',
        )
        x = self._stage(
            x,
            widths[3],
            block_count,
            stride=2,
            train=train,
            name='layer3',
        )

        # Pool final feature maps and classify: (batch, 8, 8, 640) -> (batch, 10).
        x = nn.BatchNorm(use_running_average=not train, name='bn')(x)
        x = nn.relu(x)
        x = jnp.mean(x, axis=(1, 2))
        logits = nn.Dense(self.num_classes, name='fc')(x)
        return logits

    def _stage(self, x, channels, blocks, stride, train, name):
        # Start each stage with the only block that may widen channels or downsample.
        for index in range(blocks):
            block_stride = stride if index == 0 else 1
            use_projection = index == 0
            block_name = f'{name}.{index}'
            x = WideBasicBlock(
                channels,
                stride=block_stride,
                dropout_rate=self.dropout_rate,
                use_projection=use_projection,
                name=block_name,
            )(x, train=train)
        return x


# Create and run a sample CIFAR-size image batch: (2, 32, 32, 3) -> (2, 10).
model = WideNet(depth=28, widen_factor=10, dropout_rate=0.0, num_classes=10)
test_input = jnp.ones((2, 32, 32, 3))
params = model.init(jax.random.PRNGKey(0), test_input, train=False)
logits = model.apply(params, test_input, train=False)

# logits: (2, 10)

# Train on a tiny synthetic CIFAR-size batch.
model = WideNet(depth=10, widen_factor=1, dropout_rate=0.0, num_classes=2)
train_images = jnp.zeros((2, 32, 32, 3))
train_images = train_images.at[0, 4:16, 4:16, :].set(1.0)
train_images = train_images.at[1, 16:28, 16:28, :].set(1.0)
train_targets = jnp.array([0, 1])
variables = model.init(jax.random.PRNGKey(1), train_images, train=False)
params = variables['params']
batch_stats = variables['batch_stats']


def train_step(params, batch_stats, inputs, targets, learning_rate=0.01):
    def loss_fn(current_params):
        current_variables = {'params': current_params, 'batch_stats': batch_stats}
        logits = model.apply(current_variables, inputs, train=False)
        one_hot_targets = jax.nn.one_hot(targets, logits.shape[-1])
        log_probs = jax.nn.log_softmax(logits, axis=-1)
        loss = -jnp.mean(jnp.sum(one_hot_targets * log_probs, axis=-1))
        return loss

    loss, grads = jax.value_and_grad(loss_fn)(params)
    params = jax.tree_util.tree_map(lambda p, g: p - learning_rate * g, params, grads)
    return params, loss


for step in range(3):
    params, loss = train_step(params, batch_stats, train_images, train_targets)

final_loss = loss